<a href="https://colab.research.google.com/github/em2tech/Machine_Learning_SJU/blob/main/HW5/Diabetes_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**HW5 Analysis: Ensemble Methods on the Pima Indians Diabetes Dataset**

This report answers the five analysis questions using the results produced by diabetes.ipynb. All results come from the same stratified 80/20 split (614 training and 154 test patients, random_state = 42). They are backed by repeated stratified 5-fold cross-validation (5 folds × 3 repeats) on the full 768 patients, run with leakage-safe pipelines.
One observation shapes the rest of this report. The single depth-4 decision tree has the highest accuracy on the single test split (0.786) but the lowest cross-validated accuracy (0.736). The test set holds only 154 patients, so one percentage point is about 1.5 patients. A single split is a noisy judge, and a high-variance model can "get lucky" on it. Cross-validation, which averages 15 different splits, shows the expected ordering: the ensembles beat the single tree, and Random Forest is the strongest overall. Most models fall within about one standard deviation of each other, so the differences among the top six are small.


**Explain how implicit missing values impact medical datasets, and justify your choice of imputation strategy when handling these zeros compared to outright row deletion.**

In this dataset, missing measurements were recorded as zeros instead of blanks. As a result, df.info() reports no missing data, even though glucose, blood pressure, skin thickness, insulin and BMI cannot be 0 in a living patient. The problem is widespread: 48.7% of insulin values and 29.6% of skin thickness values are placeholders, and 49% of patients have at least one missing measurement. Left as zeros, these values distort summary statistics (raw mean insulin is 79.8 instead of 155.5), corrupt feature scaling, and lead models to learn misleading patterns that reflect testing habits rather than physiology. The missingness is also not random: patients with missing values are slightly more likely to be diabetic (36.7% vs. 33.2%). Deleting incomplete rows would therefore discard about half the data, bias the sample, and leave no way to score future patients with missing tests. Median imputation, fitted only on the training data within a pipeline, avoids these problems and suits the right-skewed features, since the median (125 for insulin) is robust to the extreme values that inflate the mean. Its main limitation is that filling many values with one constant reduces variance. Adding missing-value indicators or using KNN or iterative imputation could improve on it, while outcome-based medians should be avoided because they leak the target.

**Contrast how hard voting aggregates predictions versus soft voting. Why do well-calibrated probability estimates matter when implementing soft voting across heterogeneous models?**

Both voting ensembles combine three different models: Logistic Regression, an RBF Support Vector Classifier, and a depth-4 Decision Tree. Hard voting takes each model's predicted label and returns the majority, so a model that is 51% sure counts as much as one that is 99% sure. Soft voting averages the models' predicted probabilities, so a very confident model can outweigh two that lean only slightly the other way. For example, probabilities of 0.45, 0.48 and 0.95 produce "non-diabetic" under hard voting but "diabetic" under soft voting (average 0.627). In our results, hard voting reached 0.740 test accuracy (0.766 CV) and soft voting 0.734 (0.758 CV). The two disagreed on only 3 of 154 test patients. Both were more stable than the single decision tree (0.736 CV), but neither clearly beat the best individual model, which is typical when base models make correlated errors on the same borderline cases. Recall for diabetic patients (0.556 hard, 0.537 soft) was well below that for non-diabetic patients (0.840), a key limitation for screening. Soft voting also depends on well-calibrated probabilities, because it averages outputs from different models as if they were on the same scale. Logistic Regression is generally well calibrated. The SVC produces no probabilities on its own and needed Platt scaling (CalibratedClassifierCV), and decision trees tend to be over-confident. An over-confident model can dominate the average and make the combined probability misleading, which matters in medicine, where risk scores may guide decisions. Calibrating models, checking calibration with the Brier score or reliability diagrams, and weighting the more reliable models help address this.

**3. What is the role of out-of-bag evaluation in Bagging classifiers, and why does an OOB score serve as an efficient alternative to traditional cross-validation splits during ensemble training?**

Bagging trains each tree on a bootstrap sample drawn with replacement from the training data, so each tree never sees about 36.8% of the training patients. These are its out-of-bag (OOB) patients. With oob_score=True, scikit-learn predicts each training patient using only the trees that never trained on that patient, and it reports the accuracy of those predictions as the OOB score. This gives an honest estimate of performance on unseen data, much like a validation score. Unlike k-fold cross-validation, which requires retraining the model k times on smaller portions of the data, OOB evaluation comes free as a by-product of training. It needs no separate validation set and no extra computation, which is valuable with only 614 training patients and makes it practical for tuning hyperparameters. In our results, the 500-tree Bagging model had an OOB score of 0.757, close to both its test accuracy (0.740) and its cross-validated accuracy (0.764). The OOB curve was also smoother than the test curve because it averages over more patients. OOB evaluation is especially useful because the ensemble's training accuracy is 1.000, which says nothing about generalization; a single full-depth tree also scores 1.000 in training but only 0.669 on the test set. OOB estimates can be slightly pessimistic for small ensembles, need enough trees for every patient to be out-of-bag at least once, and cannot assess preprocessing choices made outside the ensemble.

**4. How does introducing random feature subsetting in Random Forests differ from standard Bagging, and how does this feature diversity impact model variance and generalization?**


**Answer:** Bagging and Random Forests both train many full-depth trees on bootstrap samples, but they differ in how each split is chosen. Bagging lets every split consider all 8 features. A Random Forest restricts each split to a random subset, here 2 features with max_features="sqrt". This matters because averaging more trees cannot remove the variance caused by correlation between them; only making the trees less alike can. In this dataset, glucose is the dominant predictor, so all 500 Bagging trees split on glucose first and were strongly correlated. In the Random Forest, first splits were spread across glucose, BMI, age, insulin and other features, and the average correlation between trees fell from 0.391 to 0.321. Each Random Forest tree is slightly weaker on its own, but the greater diversity lets averaging remove more variance. Both ensembles improved greatly on a single deep tree, raising test accuracy from 0.669 to about 0.74. The Random Forest did slightly better on the more reliable measures, with an OOB score of 0.765 against 0.757 and the best cross-validated accuracy of all models (0.770 vs. 0.764 for Bagging), though it scored one patient lower on the small test split. Feature subsetting also produced a more informative importance ranking: glucose, BMI, age and family history ranked highest, which matches clinical understanding of diabetes risk. Insulin and skin thickness likely ranked lower partly because so many of their values were imputed.

**5. Discuss the mechanics of sequential sample weighting in AdaBoost. Based on your training versus test accuracy results, did AdaBoost exhibit signs of overfitting, and how can hyperparameter adjustments mitigate this?**

**Answer:** AdaBoost builds its ensemble sequentially from decision stumps, which are trees with a single split. Every training patient starts with equal weight. After each stump is trained, it gets a say in the final vote based on its accuracy, and the patients it misclassified are given more weight, so the next stump concentrates on the hardest cases. The final prediction is a weighted vote of all stumps. This makes boosting a bias-reduction method: simple stumps combine into a complex decision boundary, the opposite of Bagging and Random Forests, which average complex trees to reduce variance. With 300 stumps and a learning rate of 0.5, AdaBoost reached 0.809 training accuracy and 0.760 test accuracy (0.760 in cross-validation), a gap of 0.050. That indicates mild overfitting: training accuracy kept rising, while test accuracy leveled off after about 20 rounds. Overfitting stayed limited because stumps are very weak learners and the learning rate shrinks each one's contribution. Using depth-3 trees instead widened the gap to 0.196. Lowering the learning rate further reduced the gap, from 0.062 at 1.0 to 0.027 at 0.1. Overfitting can be reduced further by using a smaller learning rate with more rounds, choosing the number of rounds by early stopping, keeping the base learners shallow, tuning hyperparameters with cross-validation, and handling noisy labels or switching to a more robust boosting method.
Overall, data quality came first: almost half the patients had hidden missing values, and handling them properly was essential to every result. The parallel ensembles reduced variance. Bagging raised a single deep tree's test accuracy from 0.669 to 0.740, and the Random Forest achieved the best cross-validated accuracy (0.770). The sequential AdaBoost model reduced bias and performed competitively, with only a modest generalization gap. Evaluation also required care, since the single decision tree scored best on the small test set but worst under cross-validation. Finally, accuracy alone is not enough for medical screening: recall for diabetic patients was only about 0.54–0.57, so a real deployment should prioritize catching more diabetic cases, for example through threshold tuning, class weighting, or choosing models by recall or F1 score.
